In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Config: path to Bitrix firstline/raw CSV
# Use repo example by default; override FL_RAW_PATH if needed
FL_RAW_PATH = Path("sheets/fl_raw_09-03.csv")

# Load Bitrix raw data
kw = dict(sep=";", low_memory=False, dtype={"ID": str}, encoding="utf-8")
try:
    fl_raw = pd.read_csv(FL_RAW_PATH, on_bad_lines="warn", **kw)
except TypeError:
    # Older pandas: use error_bad_lines / warn_bad_lines
    fl_raw = pd.read_csv(FL_RAW_PATH, error_bad_lines=False, warn_bad_lines=True, **kw)

print("fl_raw shape:", fl_raw.shape)
fl_raw[["Дата создания", "UTM Medium", "ID"]].head()

fl_raw shape: (12430, 466)


,Дата создания,UTM Medium,ID
0,15.03.2026 18:48:40,description,143924
1,15.03.2026 15:40:23,context,143903
2,15.03.2026 12:53:21,NaN,143879
3,15.03.2026 11:52:52,NaN,143873
4,14.03.2026 19:20:16,NaN,143831


In [2]:
#find type of fl_raw["UTM Medium"]
fl_raw["UTM Medium"].iloc[2]

nan

In [3]:
import pandas as pd

def check_lead_type(row):
    """Check if the lead is qualified. Accepts fl_Воронка/fl_Стадия сделки or Воронка/Стадия сделки (Bitrix raw)."""
    def _scalar(v):
        if v is None:
            return None
        if hasattr(v, "iloc") and hasattr(v, "dropna"):
            v = v.dropna()
            if len(v) == 0:
                return None
            v = v.iloc[0]
        try:
            if pd.isna(v):
                return None
        except (ValueError, TypeError):
            pass
        s = str(v).strip()
        return s if s and s.lower() != "nan" else None
    funnel_type = _scalar(row.get("fl_Воронка")) or _scalar(row.get("Воронка"))
    deal_stage = _scalar(row.get("fl_Стадия сделки")) or _scalar(row.get("Стадия сделки"))
    if funnel_type in ["Входящие лиды", "Горячая воронка", "Холодная воронка"]:
        if deal_stage == "Некачественный лид":
            lead_type = "unqual"
            return lead_type
        elif deal_stage == "Получившие демо-доступ":
            lead_type = "qual"
            return lead_type
        else:
            lead_type = "unknown"
            return lead_type
    if funnel_type == "B2C" or funnel_type == "B2B":
        if deal_stage == "Сделка не заключена":
            lead_type = "refusal"
            return lead_type
        else:
            lead_type = "qual"
            return lead_type
            
    if funnel_type == "Реактивация":
        if deal_stage == "Некачественный лид":
            lead_type = "unqual"
            return lead_type
        else:
            lead_type = "unknown"
            return lead_type
    return "ignore"

In [4]:
# Add Месяц from "Дата создания" and normalize UTM Medium

month_names = {
    1: "Январь", 2: "Февраль", 3: "Март", 4: "Апрель",
    5: "Май", 6: "Июнь", 7: "Июль", 8: "Август",
    9: "Сентябрь", 10: "Октябрь", 11: "Ноябрь", 12: "Декабрь",
}

# Parse Bitrix creation date (dd.mm.yyyy HH:MM:SS, day-first)
created_dt = pd.to_datetime(fl_raw["Дата создания"], errors="coerce", dayfirst=True)
months = created_dt.dt.month
years = created_dt.dt.year

fl_raw["Месяц"] = np.where(
    created_dt.notna(),
    [f"{month_names.get(m, '')}, {y}" for m, y in zip(months, years)],
    "-",
)
fl_raw["Месяц"] = fl_raw["Месяц"].astype(str).str.strip()

# Normalize UTM Medium and UTM Source
utm_medium_col = "UTM Medium"
utm_source_col = "UTM Source"
# Verification: count empty/null UTM Medium before normalization
_before = fl_raw[utm_medium_col].copy()
_empty_before = _before.isna() | (_before.astype(str).str.strip().isin(["", "nan", "None", "-", "none", "NaN"]))
print("UTM Medium before norm: empty/null count =", _empty_before.sum(), "of", len(fl_raw))

fl_raw[utm_medium_col] = (
    fl_raw.get(utm_medium_col, pd.Series([""] * len(fl_raw), index=fl_raw.index))
    .fillna("")
    .astype(str)
    .str.strip()
)
# Replace any empty-like value with "Undeclared" (handles NaN-turned-string and blanks)
_blank = fl_raw[utm_medium_col].isin(["", "nan", "None", "-", "none", "NaN"])
fl_raw.loc[_blank, utm_medium_col] = "Undeclared"

# Verification: after normalization
print("UTM Medium after norm: 'Undeclared' count =", (fl_raw[utm_medium_col] == "Undeclared").sum())
print("UTM Medium value_counts (top 15):\n", fl_raw[utm_medium_col].value_counts().head(15))
if utm_source_col in fl_raw.columns:
    fl_raw[utm_source_col] = fl_raw[utm_source_col].astype(str).str.strip()
    fl_raw.loc[fl_raw[utm_source_col].isin(["", "nan", "None"]), utm_source_col] = "-"
    # Normalize for grouping: Sendsay/sendsay -> "sendsay"; y* (except yah*) and yd -> "yd"
    def _is_sendsay(s):
        return (s or "").strip().lower() == "sendsay"
    def _is_yandex(s):
        s = (s or "").strip().lower()
        if s == "yd":
            return True
        if s.startswith("yah"):
            return False
        return s.startswith("y")
    def _norm_src(v):
        x = str(v).strip() if pd.notna(v) else ""
        if not x or x.lower() in ("-", "nan", "none"):
            return "-"
        if _is_sendsay(x):
            return "sendsay"
        if _is_yandex(x):
            return "yd"
        return x
    fl_raw[utm_source_col] = fl_raw[utm_source_col].apply(_norm_src)
    # Normalize Yandex join identifiers (used later to filter Yandex expansions per UTM Medium)
    for _col in ["UTM Content", "UTM Term"]:
        if _col in fl_raw.columns:
            fl_raw[_col] = (
                fl_raw[_col]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
            )
            fl_raw.loc[fl_raw[_col].isin(["", "nan", "None", "-", "none"]), _col] = ""
else:
    fl_raw[utm_source_col] = "-"

# Payment metrics
# - deal_sum: numeric version of "Сумма" (invalid/missing -> 0)
# - sum_amount: deal_sum only for (Воронка in {B2B,B2C} AND Стадия сделки in {Сделка заключена, Рассрочка})
# (Paid Sum removed as redundant)
sum_col = "Сумма"
if sum_col in fl_raw.columns:
    _sum_s = fl_raw[sum_col].astype(str).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
    fl_raw["deal_sum"] = pd.to_numeric(_sum_s, errors="coerce").fillna(0)
else:
    fl_raw["deal_sum"] = 0
# Define Sum eligibility: funnel B2B/B2C + stage in {Сделка заключена, Рассрочка}
def _scalar(v):
    if v is None:
        return None
    if hasattr(v, "iloc") and hasattr(v, "dropna"):
        v = v.dropna()
        if len(v) == 0:
            return None
        v = v.iloc[0]
    try:
        if pd.isna(v):
            return None
    except (ValueError, TypeError):
        pass
    s = str(v).strip()
    return s if s and s.lower() != "nan" else None

funnel_s = fl_raw.apply(lambda r: _scalar(r.get("fl_Воронка")) or _scalar(r.get("Воронка")) or "", axis=1)
stage_s = fl_raw.apply(lambda r: _scalar(r.get("fl_Стадия сделки")) or _scalar(r.get("Стадия сделки")) or "", axis=1)
is_sum = funnel_s.astype(str).str.strip().isin(["B2B", "B2C"]) & stage_s.astype(str).str.strip().isin(["Сделка заключена", "Рассрочка","Постоплата"])
fl_raw["sum_amount"] = np.where(is_sum, fl_raw["deal_sum"], 0)


# Exclude funnels from all downstream metrics
if ("Воронка" in fl_raw.columns) or ("fl_Воронка" in fl_raw.columns):
    _f = funnel_s.astype(str).str.strip()
    fl_raw = fl_raw[~_f.isin(["Спец. проекты", "Учебный центр"])].copy()

# lead_type: check_lead_type reads Воронка/Стадия сделки (Bitrix) or fl_* (email join); handles duplicate column names
if "Воронка" in fl_raw.columns or "Стадия сделки" in fl_raw.columns or "fl_Воронка" in fl_raw.columns:
    fl_raw["lead_type"] = fl_raw.apply(check_lead_type, axis=1)
else:
    fl_raw["lead_type"] = "ignore"

fl_raw[["Месяц", "Дата создания", utm_medium_col, utm_source_col]].head()

UTM Medium before norm: empty/null count = 6941 of 12430
UTM Medium after norm: 'Undeclared' count = 6941
UTM Medium value_counts (top 15):
 UTM Medium
Undeclared        6941
context           1325
email             1053
undefined          732
social             652
bsoc_article       314
tg                 291
yanvar_context     235
cpc                209
hacker             139
search              85
letter              55
stepik              37
yanvar_search       32
anons1-19-09        23
Name: count, dtype: int64


/var/folders/m3/xhk3vyq907772brwvlh5fv280000gq/T/ipykernel_58617/1709747180.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fl_raw["Месяц"] = np.where(
/var/folders/m3/xhk3vyq907772brwvlh5fv280000gq/T/ipykernel_58617/1709747180.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fl_raw["deal_sum"] = pd.to_numeric(_sum_s, errors="coerce").fillna(0)
/var/folders/m3/xhk3vyq907772brwvlh5fv280000gq/T/ipykernel_58617/1709747180.py:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fr

,Месяц,Дата создания,UTM Medium,UTM Source
0,"Март, 2026",15.03.2026 18:48:40,description,madtest
3,"Март, 2026",15.03.2026 11:52:52,Undeclared,-
5,"Март, 2026",14.03.2026 14:52:52,Undeclared,-
6,"Март, 2026",14.03.2026 14:16:59,Undeclared,-
7,"Март, 2026",14.03.2026 12:03:19,Undeclared,-


In [5]:
# Aggregate by (Месяц, UTM Medium, UTM Source)

GROUP_MONTH_COL = "Месяц"
GROUP_MEDIUM_COL = "UTM Medium"
GROUP_SOURCE_COL = "UTM Source"

work = fl_raw.copy()

# Keep rows that have some month label ("-" kept as a separate bucket)
work[GROUP_MONTH_COL] = work[GROUP_MONTH_COL].astype(str).str.strip()
work[GROUP_MEDIUM_COL] = work[GROUP_MEDIUM_COL].astype(str).str.strip()
work.loc[work[GROUP_MEDIUM_COL].isin(["", "nan", "NaN", "None", "none", "-"]), GROUP_MEDIUM_COL] = "Undeclared"
work[GROUP_SOURCE_COL] = work.get(GROUP_SOURCE_COL, "-").astype(str).str.strip()
work.loc[work[GROUP_SOURCE_COL].isin(["", "nan", "None"]), GROUP_SOURCE_COL] = "-"

# Leads: number of rows; Qual, Unqual, Refusal, Unassigned from lead_type
agg_dict = {
    "Leads": ("ID", "size") if "ID" in work.columns else (GROUP_MONTH_COL, "size"),
    "Qual": ("lead_type", lambda s: (s == "qual").sum()),
    "Unqual": ("lead_type", lambda s: (s.isin(["unqual", "unknown"])).sum()),
    "Refusal": ("lead_type", lambda s: (s == "refusal").sum()),
    "Unassigned": ("lead_type", lambda s: (s == "ignore").sum()),
    "Sum": ("sum_amount", "sum") if "sum_amount" in work.columns else (GROUP_MONTH_COL, lambda s: 0),
}
if "lead_type" in work.columns:
    agg = (
        work.groupby([GROUP_MONTH_COL, GROUP_MEDIUM_COL, GROUP_SOURCE_COL], dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )
else:
    if "ID" in work.columns:
        agg = (
            work.groupby([GROUP_MONTH_COL, GROUP_MEDIUM_COL, GROUP_SOURCE_COL], dropna=False)
            .agg(Leads=("ID", "size"))
            .reset_index()
        )
    else:
        agg = work.groupby([GROUP_MONTH_COL, GROUP_MEDIUM_COL, GROUP_SOURCE_COL], dropna=False).size().reset_index(name="Leads")
    agg["Qual"] = 0
    agg["Unqual"] = 0
    agg["Refusal"] = 0
    agg["Unassigned"] = agg["Leads"]

agg.head(20)

,Месяц,UTM Medium,UTM Source,Leads,Qual,Unqual,Refusal,Unassigned,Sum
0,"Август, 2023",email,email,77,0,65,11,1,0.0
1,"Август, 2023",expectativa,email,4,0,4,0,0,0.0
2,"Август, 2023",mail,email,1,0,1,0,0,0.0
3,"Август, 2023",tg_cyber-ed,social,1,0,1,0,0,0.0
4,"Август, 2023",undefined,undefined,15,0,9,6,0,0.0
5,"Август, 2023",workshop,partner,4,0,0,4,0,0.0
6,"Август, 2024",Undeclared,-,4,0,0,4,0,0.0
7,"Август, 2024",undefined,undefined,2,1,1,0,0,63600.0
8,"Август, 2025",11_08,email,4,0,1,3,0,0.0
9,"Август, 2025",13_08,email,1,0,1,0,0,0.0


In [6]:
# Build Yandex and Email metric tables for drilldown (Month → campaign level).
# Use existing hierarchy exports so we also get Qual/Unqual/Refusal for drilldown rows.

from pathlib import Path

BASE_DIR = Path(".")

# Full hierarchies (all levels) for Sheets: expandable Source shows full table (header + all rows per month)
yd_hierarchy_full = pd.DataFrame()
email_hierarchy_full = pd.DataFrame()
try:
    _yd = pd.read_csv(BASE_DIR / "yd_hierarchy.csv")
    if _yd.columns[0] == "Unnamed: 0" or str(_yd.columns[0]).strip() == "":
        _yd = _yd.drop(columns=[_yd.columns[0]])
    yd_hierarchy_full = _yd
    if "Месяц" in yd_hierarchy_full.columns:
        yd_hierarchy_full["Месяц"] = yd_hierarchy_full["Месяц"].astype(str).str.strip()
    # Drop columns we don't want in the embedded Yandex table
    for _c in ["Название группы", "QPA", "fl_IDs"]:
        if _c in yd_hierarchy_full.columns:
            yd_hierarchy_full = yd_hierarchy_full.drop(columns=[_c])
    # Percent columns to 1 decimal
    for _c in ["%Qual", "%Unqual", "%Refusal"]:
        if _c in yd_hierarchy_full.columns:
            yd_hierarchy_full[_c] = pd.to_numeric(yd_hierarchy_full[_c], errors="coerce").round(1)
except Exception:
    pass
try:
    _em = pd.read_csv(BASE_DIR / "email_hierarchy_by_send.csv")
    if _em.columns[0] == "Unnamed: 0" or str(_em.columns[0]).strip() == "":
        _em = _em.drop(columns=[_em.columns[0]])
    email_hierarchy_full = _em
    if "Месяц" in email_hierarchy_full.columns:
        email_hierarchy_full["Месяц"] = email_hierarchy_full["Месяц"].astype(str).str.strip()
    if "fl_IDs" in email_hierarchy_full.columns:
        email_hierarchy_full = email_hierarchy_full.drop(columns=["fl_IDs"])
except Exception:
    pass

# --- Yandex (from yd_hierarchy.csv): Level=Campaign, keyed by (Месяц, № Кампании) ---
yandex_metrics_month_campaign = pd.DataFrame()
try:
    yd_h = pd.read_csv(BASE_DIR / "yd_hierarchy.csv")
    if "Level" in yd_h.columns:
        yd_h = yd_h[yd_h["Level"].astype(str).str.strip() == "Campaign"].copy()
    # Keep only the columns we need (if present)
    keep = [
        "Месяц",
        "№ Кампании",
        "Leads",
        "Qual",
        "Unqual",
        "Refusal",
        "Расход, ₽",
        "Клики",
        # "fl_IDs" removed
    ]
    yandex_metrics_month_campaign = yd_h[[c for c in keep if c in yd_h.columns]].copy()

    # Normalize join keys
    if "Месяц" in yandex_metrics_month_campaign.columns:
        yandex_metrics_month_campaign["Месяц"] = yandex_metrics_month_campaign["Месяц"].astype(str).str.strip()
    if "№ Кампании" in yandex_metrics_month_campaign.columns:
        yandex_metrics_month_campaign["№ Кампании"] = (
            yandex_metrics_month_campaign["№ Кампании"]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

    # Ensure lead-metric columns exist (yd_hierarchy has no Unassigned)
    for c in ["Leads", "Qual", "Unqual", "Refusal"]:
        if c not in yandex_metrics_month_campaign.columns:
            yandex_metrics_month_campaign[c] = 0
    yandex_metrics_month_campaign["Unassigned"] = 0

    # Coerce numeric metrics
    for c in ["Leads", "Qual", "Unqual", "Refusal", "Расход, ₽", "Клики"]:
        if c in yandex_metrics_month_campaign.columns:
            yandex_metrics_month_campaign[c] = pd.to_numeric(yandex_metrics_month_campaign[c], errors="coerce").fillna(0)

    # fl_IDs removed from drilldown metrics
except Exception:
    yandex_metrics_month_campaign = pd.DataFrame(
        columns=["Месяц", "№ Кампании", "Leads", "Qual", "Unqual", "Refusal", "Unassigned", "Расход, ₽", "Клики"]
    )

# --- Email (from email_hierarchy_by_send.csv): Level=Send, keyed by (Месяц, utm_campaign) ---
email_sends_month_campaign = pd.DataFrame()
try:
    em_h = pd.read_csv(BASE_DIR / "email_hierarchy_by_send.csv")
    if "Level" in em_h.columns:
        em_h = em_h[em_h["Level"].astype(str).str.strip() == "Send"].copy()

    keep = [
        "Месяц",
        "utm_campaign",
        "Leads",
        "Qual",
        "Unqual",
        "Refusal",
        "sends",
        # "fl_IDs" removed
    ]
    email_sends_month_campaign = em_h[[c for c in keep if c in em_h.columns]].copy()

    # Normalize join keys
    if "Месяц" in email_sends_month_campaign.columns:
        email_sends_month_campaign["Месяц"] = email_sends_month_campaign["Месяц"].astype(str).str.strip()
    if "utm_campaign" in email_sends_month_campaign.columns:
        email_sends_month_campaign["utm_campaign"] = (
            email_sends_month_campaign["utm_campaign"]
            .fillna("-")
            .astype(str)
            .str.strip()
        )
        email_sends_month_campaign.loc[
            email_sends_month_campaign["utm_campaign"].isin(["", "nan", "None"]),
            "utm_campaign",
        ] = "-"

    # Ensure metric columns exist (email_hierarchy_by_send has no Unassigned)
    for c in ["Leads", "Qual", "Unqual", "Refusal"]:
        if c not in email_sends_month_campaign.columns:
            email_sends_month_campaign[c] = 0
    email_sends_month_campaign["Unassigned"] = 0

    if "sends" not in email_sends_month_campaign.columns:
        # Fallback: each row is a send
        email_sends_month_campaign["sends"] = 1

    for c in ["Leads", "Qual", "Unqual", "Refusal", "sends"]:
        if c in email_sends_month_campaign.columns:
            email_sends_month_campaign[c] = pd.to_numeric(email_sends_month_campaign[c], errors="coerce").fillna(0)

    # fl_IDs removed from drilldown metrics
except Exception:
    email_sends_month_campaign = pd.DataFrame(
        columns=["Месяц", "utm_campaign", "Leads", "Qual", "Unqual", "Refusal", "Unassigned", "sends"]
    )

print("yandex_metrics_month_campaign rows:", len(yandex_metrics_month_campaign))
print("email_sends_month_campaign rows:", len(email_sends_month_campaign))
print("yd_hierarchy_full rows:", len(yd_hierarchy_full), "cols:", len(yd_hierarchy_full.columns) if len(yd_hierarchy_full) else 0)
print("email_hierarchy_full rows:", len(email_hierarchy_full), "cols:", len(email_hierarchy_full.columns) if len(email_hierarchy_full) else 0)

yandex_metrics_month_campaign rows: 101
email_sends_month_campaign rows: 85
yd_hierarchy_full rows: 360 cols: 15
email_hierarchy_full rows: 99 cols: 17


In [7]:
# Build hierarchy rows: Month / Medium / Source / [MetricHeader, MetricRow]

LEVEL_MONTH = "Month"
LEVEL_MEDIUM = "Medium"
LEVEL_SOURCE = "Source"
LEVEL_METRIC_HEADER = "MetricHeader"
LEVEL_METRIC_ROW = "MetricRow"
# (No Spacer rows; Month row already represents the month total)

# Source → metrics: normalize matching to follow your tagging rules.
# - Sendsay and sendsay are the same
# - Anything starting with "y" is Yandex, except "yah..." (e.g. "yah" is NOT Yandex)
# - Keep legacy "yd" as Yandex too
def _is_sendsay_source(src_lower: str) -> bool:
    return src_lower == "sendsay"

def _is_yandex_source(src_lower: str) -> bool:
    if src_lower == "yd":
        return True
    if src_lower.startswith("yah"):
        return False
    return src_lower.startswith("y")

# Optional metric tables (built in next cell); keyed by Месяц for drilldown. If absent, no metric blocks.
yandex_metrics = globals().get("yandex_metrics_month_campaign", pd.DataFrame())
email_sends_metrics = globals().get("email_sends_month_campaign", pd.DataFrame())

def _empty_row(level, month_val, medium_val, source_val, campaign_val="-"):
    return {
        "Level": level,
        GROUP_MONTH_COL: month_val,
        GROUP_MEDIUM_COL: medium_val,
        GROUP_SOURCE_COL: source_val,
        "Campaign": campaign_val,
        "Leads": 0,
        "Qual": 0,
        "Unqual": 0,
        "Refusal": 0,
        "Unassigned": 0,
        "Sum": 0,
        # fl_IDs removed
        "Расход, ₽": 0,
        "Клики": 0,
        "sends": 0,
    }

# Helper: sort month labels chronologically (then we will reverse)
month_name_to_num = {
    "Январь": 1,
    "Февраль": 2,
    "Март": 3,
    "Апрель": 4,
    "Май": 5,
    "Июнь": 6,
    "Июль": 7,
    "Август": 8,
    "Сентябрь": 9,
    "Октябрь": 10,
    "Ноябрь": 11,
    "Декабрь": 12,
}

def _month_sort_key(label: str):
    s = str(label).strip()
    if not s or s == "-":
        return (0, 0)
    parts = [p.strip() for p in s.split(",")]
    if len(parts) != 2:
        return (0, 0)
    name, year_s = parts[0], parts[1]
    try:
        y = int(year_s)
    except (ValueError, TypeError):
        return (0, 0)
    m = month_name_to_num.get(name, 0)
    return (y, m)

# Distinct months in reverse chronological order
months_unique = (
    agg[GROUP_MONTH_COL]
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)
months_sorted = sorted(months_unique, key=_month_sort_key, reverse=True)

rows_out = []
for month in months_sorted:
    m_slice = agg[agg[GROUP_MONTH_COL].astype(str).str.strip() == month]
    if m_slice.empty:
        continue
    # Ensure missing/blank medium buckets always roll up under a single label
    m_slice[GROUP_MEDIUM_COL] = m_slice[GROUP_MEDIUM_COL].astype(str).str.strip()
    m_slice.loc[m_slice[GROUP_MEDIUM_COL].isin(["", "nan", "NaN", "None", "none", "-"]), GROUP_MEDIUM_COL] = "Undeclared"

    month_total_leads = int(m_slice["Leads"].sum())
    month_qual = int(m_slice["Qual"].sum()) if "Qual" in m_slice.columns else 0
    month_unqual = int(m_slice["Unqual"].sum()) if "Unqual" in m_slice.columns else 0
    month_refusal = int(m_slice["Refusal"].sum()) if "Refusal" in m_slice.columns else 0
    month_unassigned = int(m_slice["Unassigned"].sum()) if "Unassigned" in m_slice.columns else 0
    # fl_IDs removed
    month_sum = float(pd.to_numeric(m_slice.get("Sum", 0), errors="coerce").fillna(0).sum())

    rows_out.append(
        {
            "Level": LEVEL_MONTH,
            GROUP_MONTH_COL: month,
            GROUP_MEDIUM_COL: "-",
            GROUP_SOURCE_COL: "-",
            "Campaign": "-",
            "Leads": month_total_leads,
            "Qual": month_qual,
            "Unqual": month_unqual,
            "Refusal": month_refusal,
            "Unassigned": month_unassigned,
            "Sum": month_sum,
            # fl_IDs removed
            "Расход, ₽": 0,
            "Клики": 0,
            "sends": 0,
        }
    )

    # Group by medium, then by source
    m_slice_sorted = m_slice.sort_values([GROUP_MEDIUM_COL, GROUP_SOURCE_COL])
    for medium_val, med_df in m_slice_sorted.groupby(GROUP_MEDIUM_COL, sort=True):
        med_leads = int(med_df["Leads"].sum())
        med_qual = int(med_df["Qual"].sum()) if "Qual" in med_df.columns else 0
        med_unqual = int(med_df["Unqual"].sum()) if "Unqual" in med_df.columns else 0
        med_refusal = int(med_df["Refusal"].sum()) if "Refusal" in med_df.columns else 0
        med_unassigned = int(med_df["Unassigned"].sum()) if "Unassigned" in med_df.columns else 0
        med_sum = float(pd.to_numeric(med_df.get("Sum", 0), errors="coerce").fillna(0).sum())
        # fl_IDs removed
        rows_out.append(
            {
                "Level": LEVEL_MEDIUM,
                GROUP_MONTH_COL: month,
                GROUP_MEDIUM_COL: medium_val,
                GROUP_SOURCE_COL: "-",
                "Campaign": "-",
                "Leads": med_leads,
                "Qual": med_qual,
                "Unqual": med_unqual,
                "Refusal": med_refusal,
                "Unassigned": med_unassigned,
                "Sum": med_sum,
                # fl_IDs removed
                "Расход, ₽": 0,
                "Клики": 0,
                "sends": 0,
            }
        )
        for _, r in med_df.iterrows():
            src_val = str(r.get(GROUP_SOURCE_COL, "-")).strip()
            rows_out.append(
                {
                    "Level": LEVEL_SOURCE,
                    GROUP_MONTH_COL: month,
                    GROUP_MEDIUM_COL: medium_val,
                    GROUP_SOURCE_COL: src_val,
                    "Campaign": "-",
                    "Leads": int(r["Leads"]),
                    "Qual": int(r.get("Qual", 0)),
                    "Unqual": int(r.get("Unqual", 0)),
                    "Refusal": int(r.get("Refusal", 0)),
                    "Unassigned": int(r.get("Unassigned", 0)),
                    "Sum": float(pd.to_numeric(r.get("Sum", 0), errors="coerce") or 0),
                    # fl_IDs removed
                    "Расход, ₽": 0,
                    "Клики": 0,
                    "sends": 0,
                }
            )
            src_lower = src_val.lower()
            if _is_yandex_source(src_lower) and len(yandex_metrics):
                month_norm = str(month).strip()
                yd_block = yandex_metrics[yandex_metrics[GROUP_MONTH_COL].astype(str).str.strip() == month_norm]
                if not yd_block.empty:
                    rows_out.append(_empty_row(LEVEL_METRIC_HEADER, month, "-", src_val))
                    for _, mr in yd_block.iterrows():
                        camp_raw = mr.get("№ Кампании", mr.get("Campaign", "-"))
                        camp = str(camp_raw).strip().replace(".0", "")
                        rows_out.append({
                            **_empty_row(LEVEL_METRIC_ROW, month, "-", src_val, camp),
                            "Leads": int(pd.to_numeric(mr.get("Leads", 0), errors="coerce") or 0),
                            "Qual": int(pd.to_numeric(mr.get("Qual", 0), errors="coerce") or 0),
                            "Unqual": int(pd.to_numeric(mr.get("Unqual", 0), errors="coerce") or 0),
                            "Refusal": int(pd.to_numeric(mr.get("Refusal", 0), errors="coerce") or 0),
                            "Unassigned": int(pd.to_numeric(mr.get("Unassigned", 0), errors="coerce") or 0),
                            # fl_IDs removed
                            "Расход, ₽": pd.to_numeric(mr.get("Расход, ₽", 0), errors="coerce") or 0,
                            "Клики": int(pd.to_numeric(mr.get("Клики", 0), errors="coerce") or 0),
                        })
            if _is_sendsay_source(src_lower) and len(email_sends_metrics):
                month_norm = str(month).strip()
                em_block = email_sends_metrics[email_sends_metrics[GROUP_MONTH_COL].astype(str).str.strip() == month_norm]
                if not em_block.empty:
                    rows_out.append(_empty_row(LEVEL_METRIC_HEADER, month, "-", src_val))
                    for _, mr in em_block.iterrows():
                        camp = str(mr.get("utm_campaign", mr.get("Campaign", "-")) or "-").strip()
                        if not camp or camp.lower() in ("nan", "none"):
                            camp = "-"
                        rows_out.append({
                            **_empty_row(LEVEL_METRIC_ROW, month, "-", src_val, camp),
                            "Leads": int(pd.to_numeric(mr.get("Leads", 0), errors="coerce") or 0),
                            "Qual": int(pd.to_numeric(mr.get("Qual", 0), errors="coerce") or 0),
                            "Unqual": int(pd.to_numeric(mr.get("Unqual", 0), errors="coerce") or 0),
                            "Refusal": int(pd.to_numeric(mr.get("Refusal", 0), errors="coerce") or 0),
                            "Unassigned": int(pd.to_numeric(mr.get("Unassigned", 0), errors="coerce") or 0),
                            # fl_IDs removed
                            "sends": int(pd.to_numeric(mr.get("sends", 0), errors="coerce") or 0),
                        })


fl_hierarchy_by_month_medium_source = pd.DataFrame(rows_out)
print("fl_hierarchy_by_month_medium_source: rows", len(fl_hierarchy_by_month_medium_source))
fl_hierarchy_by_month_medium_source.head(40)

fl_hierarchy_by_month_medium_source: rows 1224


,Level,Месяц,UTM Medium,UTM Source,Campaign,Leads,Qual,Unqual,Refusal,Unassigned,Sum,"Расход, ₽",Клики,sends
0,Month,"Март, 2026",-,-,-,546,131,404,10,1,1078651.52,0,0,0
1,Medium,"Март, 2026",%7Bsource_type%7D,-,-,1,0,1,0,0,0.00,0,0,0
2,Source,"Март, 2026",%7Bsource_type%7D,yd,-,1,0,1,0,0,0.00,0,0,0
3,MetricHeader,"Март, 2026",-,yd,-,0,0,0,0,0,0.00,0,0,0
4,MetricRow,"Март, 2026",-,yd,707144191,0,0,0,0,0,0.00,16681,107,0
5,MetricRow,"Март, 2026",-,yd,707219401,0,0,0,0,0,0.00,13564,194,0
6,MetricRow,"Март, 2026",-,yd,707265139,0,0,0,0,0,0.00,4167,55,0
7,MetricRow,"Март, 2026",-,yd,707634725,21,4,17,0,0,0.00,9733,239,0
8,MetricRow,"Март, 2026",-,yd,707634735,28,4,24,0,0,0.00,11177,258,0
9,MetricRow,"Март, 2026",-,yd,707634746,22,3,19,0,0,0.00,11403,236,0


In [8]:
# Build sheet table for Google Sheets: 11 Bitrix cols + M drilldown cols.
# Main header = Bitrix only. Under each Yandex/Email Source: hierarchy sub-header + all hierarchy rows for that month.

BITRIX_COLS = ["Level", "Месяц", "UTM Medium", "UTM Source", "Leads", "Qual", "Unqual", "Refusal", "Unassigned", "Sum"]
def _is_sendsay_source(src_lower: str) -> bool:
    return src_lower == "sendsay"

def _is_yandex_source(src_lower: str) -> bool:
    if src_lower == "yd":
        return True
    if src_lower.startswith("yah"):
        return False
    return src_lower.startswith("y")

# Drilldown width = max of the two hierarchy column counts
M = 0
yd_cols = []
yd_drill_cols = []
em_cols = []
if len(yd_hierarchy_full) > 0:
    yd_cols = list(yd_hierarchy_full.columns)
    yd_drill_cols = yd_cols + ["Profit"]
    M = max(M, len(yd_drill_cols))
if len(email_hierarchy_full) > 0:
    em_cols = list(email_hierarchy_full.columns)
    M = max(M, len(em_cols))
if M == 0:
    M = 1

sheet_data = []
# Row 0: Bitrix header (11) + M empty
sheet_data.append(BITRIX_COLS + [""] * M)

# Row-group ranges for Yandex campaigns (0-based, end-exclusive, includes header row in indexing)
yandex_campaign_groups = []

# Build per-(month, medium) Yandex id sets from Bitrix (UTM Source == 'yd')
MONTH_COL = "Месяц"
MED_COL = "UTM Medium"
SRC_COL = "UTM Source"
CONTENT_COL = "UTM Content"  # often ad id
TERM_COL = "UTM Term"        # often campaign id

medium_to_ad_ids = {}
medium_to_campaign_ids = {}

# Map Bitrix sums → Yandex spend rows via ad_id → campaign_id
ad_to_campaign = {}
sum_by_campaign = {}
sum_by_month_medium = {}
try:
    _w = fl_raw.copy()
    for _c in [MONTH_COL, MED_COL, SRC_COL]:
        if _c in _w.columns:
            _w[_c] = _w[_c].fillna("").astype(str).str.strip()
    _w = _w[_w[SRC_COL].astype(str).str.strip().str.lower() == "yd"]
    if CONTENT_COL in _w.columns:
        _w[CONTENT_COL] = _w[CONTENT_COL].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if TERM_COL in _w.columns:
        _w[TERM_COL] = _w[TERM_COL].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "deal_sum" in _w.columns:
        _w["deal_sum"] = pd.to_numeric(_w["deal_sum"], errors="coerce").fillna(0)
    else:
        _w["deal_sum"] = 0
    # Paid Sum removed; only sum_amount is used for profit

    # Build ad_id → campaign_id mapping from Yandex hierarchy (Ad-level rows)
    if len(yd_hierarchy_full) > 0 and ("№ Объявления" in yd_hierarchy_full.columns) and ("№ Кампании" in yd_hierarchy_full.columns):
        _admap = yd_hierarchy_full[["№ Объявления", "№ Кампании"]].copy()
        _admap["№ Объявления"] = _admap["№ Объявления"].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        _admap["№ Кампании"] = _admap["№ Кампании"].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        _admap = _admap[(_admap["№ Объявления"] != "") & (_admap["№ Объявления"] != "-") & (_admap["№ Кампании"] != "") & (_admap["№ Кампании"] != "-")]
        for _, rr in _admap.drop_duplicates(subset=["№ Объявления"]).iterrows():
            ad_to_campaign[str(rr["№ Объявления"]).strip()] = str(rr["№ Кампании"]).strip()

    # Normalize Bitrix UTM Content to a real Yandex ad_id (handles prefixes/text)
    import re
    def _extract_ad_id(v):
        s = str(v).strip() if v is not None else ""
        if not s or s.lower() in ("nan", "none", "-"):
            return ""
        parts = re.findall(r"\d{5,}", s)
        if not parts:
            return ""
        return max(parts, key=len)

    if CONTENT_COL in _w.columns:
        _w["ad_id"] = _w[CONTENT_COL].apply(_extract_ad_id)
    else:
        _w["ad_id"] = ""
    _w["campaign_id"] = _w["ad_id"].map(ad_to_campaign).fillna("")

    # Per-(month, medium) totals + id sets (for filtering)
    for (m, med), g in _w.groupby([MONTH_COL, MED_COL], dropna=False):
        key = (str(m).strip(), str(med).strip())
        sum_by_month_medium[key] = float(pd.to_numeric(g.get("sum_amount", 0), errors="coerce").fillna(0).sum())
        s = set(g["ad_id"].dropna().astype(str).str.strip())
        s = {x for x in s if x and x.lower() not in ("nan", "none", "-")}
        if s:
            medium_to_ad_ids[key] = s
        s2 = set(g["campaign_id"].dropna().astype(str).str.strip())
        s2 = {x for x in s2 if x and x.lower() not in ("nan", "none", "-")}
        if s2:
            medium_to_campaign_ids[key] = s2

    # Aggregate Bitrix sums per (month, medium, campaign_id)
    _wc = _w[_w["campaign_id"].astype(str).str.strip() != ""]
    if len(_wc) > 0:
        _camp_sum = (
            _wc.groupby([MONTH_COL, MED_COL, "campaign_id"], dropna=False)[["sum_amount"]]
            .sum()
            .reset_index()
        )
        for _, rr in _camp_sum.iterrows():
            k = (str(rr[MONTH_COL]).strip(), str(rr[MED_COL]).strip(), str(rr["campaign_id"]).strip())
            sum_by_campaign[k] = float(rr["sum_amount"])
except Exception:
    pass

# Normalize Yandex hierarchy id columns once for reliable matching
if len(yd_hierarchy_full) > 0:
    if "№ Объявления" in yd_hierarchy_full.columns:
        yd_hierarchy_full["№ Объявления"] = yd_hierarchy_full["№ Объявления"].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "№ Кампании" in yd_hierarchy_full.columns:
        yd_hierarchy_full["№ Кампании"] = yd_hierarchy_full["№ Кампании"].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

print("[yandex-per-medium] keys with ad_ids:", len(medium_to_ad_ids), "keys with campaign_ids:", len(medium_to_campaign_ids))
print("[yandex-profit] ad_to_campaign:", len(ad_to_campaign), "sum_by_campaign:", len(sum_by_campaign), "sum_by_month_medium:", len(sum_by_month_medium))
try:
    _nz = sum(1 for _k, (s1, s2) in sum_by_campaign.items() if (s1 or 0) != 0 or (s2 or 0) != 0)
    print("[yandex-profit] campaigns with nonzero sums:", _nz)
except Exception:
    pass
try:
    # quick sample check: show one key's totals
    _k = next(iter(sum_by_month_medium.keys())) if len(sum_by_month_medium) else None
    if _k:
        print("[yandex-profit] sample month/medium:", _k, "sum/paid:", sum_by_month_medium.get(_k))
except Exception:
    pass

df = fl_hierarchy_by_month_medium_source
i = 0
while i < len(df):
    row = df.iloc[i]
    level = str(row.get("Level", "")).strip()
    # Bitrix segment: take first 11 columns from the hierarchy row (in order of BITRIX_COLS)
    bitrix_vals = [str(row.get(c, "")).strip() if c in row.index else "" for c in BITRIX_COLS]

    if level in ("Month", "Medium"):
        sheet_data.append(bitrix_vals + [""] * M)
        i += 1
        continue

    if level == "Source":
        sheet_data.append(bitrix_vals + [""] * M)
        src = str(row.get("UTM Source", "")).strip().lower()
        month_val = str(row.get("Месяц", "")).strip()

        if _is_yandex_source(src) and len(yd_hierarchy_full) > 0:
            # Yandex sub-header: empty Bitrix segment, then yd columns + profit columns (padded to M)
            drill_header = list(yd_drill_cols) + [""] * (M - len(yd_drill_cols))
            sheet_data.append([""] * len(BITRIX_COLS) + drill_header)
            medium_val = str(row.get("UTM Medium", "")).strip()
            key = (month_val, medium_val)
            ad_ids = medium_to_ad_ids.get(key, set())
            camp_ids = medium_to_campaign_ids.get(key, set())
            yd_month = yd_hierarchy_full[yd_hierarchy_full["Месяц"].astype(str).str.strip() == month_val]
            # Important: filter by campaign whenever possible so Campaign rows remain present for grouping
            camp_ids_from_ads = set()
            if ad_ids:
                camp_ids_from_ads = {ad_to_campaign.get(a, "") for a in ad_ids}
                camp_ids_from_ads = {c for c in camp_ids_from_ads if c and str(c).strip().lower() not in ("nan", "none", "-")}
            if camp_ids_from_ads and ("№ Кампании" in yd_month.columns):
                yd_month = yd_month[yd_month["№ Кампании"].astype(str).str.strip().isin(camp_ids_from_ads)]
            elif camp_ids and ("№ Кампании" in yd_month.columns):
                yd_month = yd_month[yd_month["№ Кампании"].astype(str).str.strip().isin(camp_ids)]
            elif ad_ids and ("№ Объявления" in yd_month.columns):
                # last resort (will likely drop Campaign rows)
                yd_month = yd_month[yd_month["№ Объявления"].astype(str).str.strip().isin(ad_ids)]
            # Track campaign → children rows for collapse
            current_child_start = None
            for _, r in yd_month.iterrows():
                y_level = str(r.get("Level", "") or "").strip()
                # Close previous campaign group when a new campaign row starts
                if y_level == "Campaign" and (current_child_start is not None) and (len(sheet_data) > current_child_start):
                    yandex_campaign_groups.append((current_child_start, len(sheet_data)))
                ad_id = str(r.get("№ Объявления", "") or "").strip().replace(".0", "")
                camp_id = str(r.get("№ Кампании", "") or "").strip().replace(".0", "")
                if ad_id.lower() in ("", "nan", "none", "-"):
                    ad_id = ""
                if camp_id.lower() in ("", "nan", "none", "-"):
                    camp_id = ""

                bitrix_sum = 0.0
                # Attribute sums by campaign (ad rows roll up via ad_to_campaign)
                if (not camp_id) and ad_id:
                    camp_id = ad_to_campaign.get(ad_id, "")
                if camp_id:
                    bitrix_sum = float(sum_by_campaign.get((month_val, medium_val, camp_id), 0.0) or 0.0)
                else:
                    bitrix_sum = float(sum_by_month_medium.get((month_val, medium_val), 0.0) or 0.0)

                spend = float(pd.to_numeric(r.get("Расход, ₽", 0), errors="coerce") or 0)
                profit = bitrix_sum - spend

                drill_vals = [str(r[c]) if c in r.index else "" for c in yd_cols] + [profit]
                drill_vals = drill_vals + [""] * (M - len(drill_vals))
                sheet_data.append([""] * len(BITRIX_COLS) + drill_vals)
                # If this is a campaign header row, the children start on the next row
                if y_level == "Campaign":
                    current_child_start = len(sheet_data)
            # Close final campaign group in this Yandex block
            if (current_child_start is not None) and (len(sheet_data) > current_child_start):
                yandex_campaign_groups.append((current_child_start, len(sheet_data)))

        if _is_sendsay_source(src) and len(email_hierarchy_full) > 0:
            drill_header = list(em_cols) + [""] * (M - len(em_cols))
            sheet_data.append([""] * len(BITRIX_COLS) + drill_header)
            em_month = email_hierarchy_full[email_hierarchy_full["Месяц"].astype(str).str.strip() == month_val]
            for _, r in em_month.iterrows():
                drill_vals = [str(r[c]) if c in r.index else "" for c in em_cols]
                drill_vals = drill_vals + [""] * (M - len(drill_vals))
                sheet_data.append([""] * len(BITRIX_COLS) + drill_vals)

        # Skip following MetricHeader and MetricRow rows in the Bitrix hierarchy
        i += 1
        while i < len(df) and str(df.iloc[i].get("Level", "")).strip() in ("MetricHeader", "MetricRow"):
            i += 1
        continue

    # MetricHeader / MetricRow: skip (replaced by full hierarchy block above)
    i += 1

# fl_sheet_table = list of rows for Sheets: row 0 = Bitrix header + M empty; row 1+ = data
fl_sheet_table = sheet_data


print("fl_sheet_table: %d rows (incl. header), %d cols" % (len(fl_sheet_table), len(fl_sheet_table[0]) if fl_sheet_table else 0))

[yandex-per-medium] keys with ad_ids: 25 keys with campaign_ids: 22
[yandex-profit] ad_to_campaign: 246 sum_by_campaign: 71 sum_by_month_medium: 37
[yandex-profit] sample month/medium: ('Апрель, 2025', 'cpc') sum/paid: 0.0
fl_sheet_table: 1352 rows (incl. header), 27 cols


In [9]:
# Optional: export the hierarchy to CSV for inspection

fl_hierarchy_by_month_medium_source.to_csv("fl_hierarchy_by_month_medium_source.csv", index=False)
print("Saved fl_hierarchy_by_month_medium_source.csv")

Saved fl_hierarchy_by_month_medium_source.csv


In [10]:
# Push Bitrix hierarchy (Month → Medium → Source) or sheet table (11 Bitrix cols + drilldown) to Google Sheets.
# Requires: pip install gspread google-auth

def push_fl_hierarchy_to_sheets(
    hierarchy_df,
    spreadsheet_id_or_url: str = None,
    worksheet_name: str = "FL by Month Medium Source",
    credentials_path: str = None,
    yandex_campaign_groups: list = None,
) -> None:
    """Write to Sheet. hierarchy_df can be a list of rows (fl_sheet_table: row 0 = header, row 1+ = data) or a DataFrame. Row groups: Month / Medium / Source (Level in col 0) + optional Yandex campaign child groups."""
    try:
        import gspread
        from google.oauth2.service_account import Credentials
    except ImportError:
        print("Install gspread and google-auth: pip install gspread google-auth")
        return

    scopes = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ]
    creds_path = credentials_path or __import__("os").environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if not creds_path:
        print("Set GOOGLE_APPLICATION_CREDENTIALS or pass credentials_path=...")
        return

    p = Path(creds_path)
    if p.is_dir():
        jsons = list(p.glob("*.json"))
        if not jsons:
            print(f"No .json in directory {creds_path}")
            return
        creds_path = str(jsons[0])
    elif "*" in str(creds_path):
        import glob
        jsons = glob.glob(creds_path)
        if not jsons:
            print(f"No file matching {creds_path}")
            return
        creds_path = jsons[0]
    elif not p.exists():
        print(f"Credentials file not found: {creds_path}")
        return

    try:
        creds = Credentials.from_service_account_file(creds_path, scopes=scopes)
        gc = gspread.authorize(creds)
    except Exception as e:
        print(f"Credentials error: {e}")
        return

    sheet_id = spreadsheet_id_or_url or __import__("os").environ.get("GOOGLE_SHEET_ID")
    if not sheet_id:
        print("Set spreadsheet_id_or_url or GOOGLE_SHEET_ID")
        return

    is_url = "/" in sheet_id and ("docs.google.com" in sheet_id or sheet_id.startswith("http"))
    if is_url:
        sheet_id_clean = sheet_id.split("/d/")[-1].split("/")[0].split("?")[0]
    else:
        sheet_id_clean = sheet_id

    try:
        sh = gc.open_by_url(sheet_id) if (is_url and hasattr(gc, "open_by_url")) else gc.open_by_key(sheet_id_clean)
    except Exception as e:
        print("Could not open spreadsheet (check ID and edit access for service account):", e)
        return

    try:
        ws = sh.worksheet(worksheet_name)
    except gspread.exceptions.WorksheetNotFound:
        num_rows = (len(hierarchy_df) + 10) if isinstance(hierarchy_df, list) else (len(hierarchy_df) + 10)
        num_cols = len(hierarchy_df[0]) if isinstance(hierarchy_df, list) and hierarchy_df else len(hierarchy_df.columns)
        ws = sh.add_worksheet(title=worksheet_name, rows=max(1000, num_rows), cols=max(20, num_cols))

    if isinstance(hierarchy_df, list):
        data = [[str(x) for x in row] for row in hierarchy_df]
        n_rows = len(data)
        level_col = 0
    else:
        df = hierarchy_df.fillna("").reset_index(drop=True)
        data = [df.columns.tolist()] + df.astype(str).values.tolist()
        n_rows = len(data)
        level_col = df.columns.get_loc("Level") if "Level" in df.columns else 0

    try:
        ws.clear()
        ws.update(data, range_name="A1", value_input_option="USER_ENTERED")
    except Exception as e:
        print(f"Could not write data: {e}")
        return

    sheet_id_val = int(ws.id)
    requests = []
    # Row 0 = header; data rows are 1..n_rows-1. Level is at level_col.
    if n_rows > 1:
        i = 1
        while i < n_rows:
            level = str(data[i][level_col]).strip() if level_col < len(data[i]) else ""
            if level != "Month":
                i += 1
                continue
            k = i + 1
            while k < n_rows and (str(data[k][level_col]).strip() if level_col < len(data[k]) else "") != "Month":
                k += 1
            end_block = k
            if end_block > i + 1:
                requests.append({
                    "addDimensionGroup": {
                        "range": {"sheetId": sheet_id_val, "dimension": "ROWS", "startIndex": i + 1, "endIndex": end_block}
                    }
                })
            j = i + 1
            while j < end_block:
                lev_j = str(data[j][level_col]).strip() if level_col < len(data[j]) else ""
                if lev_j == "Medium":
                    end_med = j + 1
                    while end_med < end_block:
                        lv = str(data[end_med][level_col]).strip() if level_col < len(data[end_med]) else ""
                        if lv in ("Medium", "Month"):
                            break
                        end_med += 1
                    if end_med > j + 1:
                        requests.append({
                            "addDimensionGroup": {
                                "range": {"sheetId": sheet_id_val, "dimension": "ROWS", "startIndex": j + 1, "endIndex": end_med}
                            }
                        })
                    q = j + 1
                    while q < end_med:
                        lv_q = str(data[q][level_col]).strip() if level_col < len(data[q]) else ""
                        if lv_q == "Source":
                            end_src = q + 1
                            while end_src < end_med:
                                lv_s = str(data[end_src][level_col]).strip() if level_col < len(data[end_src]) else ""
                                if lv_s in ("Source", "Medium", "Month"):
                                    break
                                end_src += 1
                            if end_src > q + 1:
                                requests.append({
                                    "addDimensionGroup": {
                                        "range": {"sheetId": sheet_id_val, "dimension": "ROWS", "startIndex": q + 1, "endIndex": end_src}
                                    }
                                })
                            q = end_src
                        else:
                            q += 1
                    j = end_med
                else:
                    j += 1
            i = k

    # Optional: add campaign-level groups inside Yandex drilldowns
    if yandex_campaign_groups:
        for (start_i, end_i) in yandex_campaign_groups:
            if end_i > start_i:
                requests.append({
                    "addDimensionGroup": {
                        "range": {"sheetId": sheet_id_val, "dimension": "ROWS", "startIndex": int(start_i), "endIndex": int(end_i)}
                    }
                })

    if requests:
        try:
            sh.batch_update({"requests": requests})
        except Exception as e:
            print(f"Row grouping failed (data was written): {e}")
        print(f"Updated sheet '{worksheet_name}', {len(requests)} row groups (expand/collapse).")
    else:
        print(f"Updated sheet '{worksheet_name}' (no row groups).")

In [11]:
GOOGLE_SHEET_ID = "https://docs.google.com/spreadsheets/d/1Q5KmqXUOVe9hVXpyJ1m1yUqPEaQ_2KXjoH71DHiPrLU/edit?gid=0#gid=0"
GOOGLE_APPLICATION_CREDENTIALS = "/Users/ivan/Documents/deved/keys/cybered-490317-a7b083e70c85.json"
credentials_path = "/Users/ivan/Documents/deved/keys/cybered-490317-a7b083e70c85.json"
import os

push_fl_hierarchy_to_sheets(
    fl_sheet_table,
    spreadsheet_id_or_url=GOOGLE_SHEET_ID,
    worksheet_name="FL with quals",
    credentials_path=credentials_path,
    yandex_campaign_groups=yandex_campaign_groups,
)

Updated sheet 'FL with quals', 456 row groups (expand/collapse).
